# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoro-369/flyrank-ml-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I will use a simple **refresh-opportunity baseline** based on two observable signals:

1. **Staleness:** the page has not been updated for at least 180 days.
2. **Visibility:** the page has received at least 500 impressions in the observed 90-day window.

The rule selects a page when it is both stale and visible, then ranks selected pages by impressions. This follows the idea behind a `stale_visible_page` refresh flag: a page that is old but still has meaningful search exposure is worth putting earlier in a human review queue.

The rule has one reason code:

- `stale_visible_page` — the page is at least 180 days old and has at least 500 impressions.

The action label is:

- `review_refresh` — send the page to a human reviewer for a refresh decision.

Pages that do not satisfy both conditions receive score `0`, reason code `not_selected`, and action `monitor`.

The signal audit below uses the observed `trend_direction` only to check whether the two signals are directionally useful. `trend_direction` and `trend_pct` are **not inputs to the baseline score**.

In [2]:
# Load the starter dataset and audit the two signals before encoding the rule.
# The notebook can be run from the repository root, work/, or work/notebooks/.

import os
import numpy as np
import pandas as pd

def find_data_path():
    candidates = [
        os.path.join(os.getcwd(), "data", "raw", "content_refresh_anonymized.csv"),
        os.path.join(os.getcwd(), "..", "data", "raw", "content_refresh_anonymized.csv"),
        os.path.join(os.getcwd(), "..", "..", "data", "raw", "content_refresh_anonymized.csv"),
    ]

    for path in candidates:
        path = os.path.abspath(path)
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Run this notebook from the FlyRank repository."
    )

DATA_PATH = find_data_path()
df = pd.read_csv(DATA_PATH)

print("Dataset:", DATA_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))

required = [
    "days_since_last_update",
    "impressions_90d",
    "trend_direction"
]

missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

# This is an observed outcome used ONLY for signal auditing.
# It is deliberately excluded from the baseline score.
df["is_declining_audit"] = (
    df["trend_direction"]
      .astype(str)
      .str.lower()
      .eq("down")
      .astype(int)
)

STALE_DAYS = 180
VISIBLE_IMPRESSIONS = 500

# ------------------------------------------------------------
# Signal 1: Staleness
# ------------------------------------------------------------
df["staleness_bucket"] = np.where(
    df["days_since_last_update"] >= STALE_DAYS,
    "stale_180_plus_days",
    "updated_within_180_days"
)

staleness_table = (
    df.groupby("staleness_bucket", dropna=False)
      .agg(
          n=("is_declining_audit", "size"),
          declining_n=("is_declining_audit", "sum"),
          declining_rate=("is_declining_audit", "mean")
      )
      .reset_index()
)

print("\n=== SIGNAL 1: STALENESS ===")
print(staleness_table.to_string(index=False))

stale_rate = staleness_table.loc[
    staleness_table["staleness_bucket"] == "stale_180_plus_days",
    "declining_rate"
].iloc[0]

fresh_rate = staleness_table.loc[
    staleness_table["staleness_bucket"] == "updated_within_180_days",
    "declining_rate"
].iloc[0]

staleness_difference = stale_rate - fresh_rate

if staleness_difference > 0.05:
    staleness_verdict = "CONFIRMED"
elif staleness_difference < -0.05:
    staleness_verdict = "OPPOSITE"
else:
    staleness_verdict = "MIXED"

print(f"Staleness verdict: {staleness_verdict}")
print(f"Observed declining-rate difference: {staleness_difference:.3f}")

# ------------------------------------------------------------
# Signal 2: Visibility
# ------------------------------------------------------------
df["visibility_bucket"] = np.where(
    df["impressions_90d"] >= VISIBLE_IMPRESSIONS,
    "visible_500_plus_impressions",
    "below_500_impressions"
)

visibility_table = (
    df.groupby("visibility_bucket", dropna=False)
      .agg(
          n=("is_declining_audit", "size"),
          declining_n=("is_declining_audit", "sum"),
          declining_rate=("is_declining_audit", "mean")
      )
      .reset_index()
)

print("\n=== SIGNAL 2: VISIBILITY ===")
print(visibility_table.to_string(index=False))

visible_rate = visibility_table.loc[
    visibility_table["visibility_bucket"] == "visible_500_plus_impressions",
    "declining_rate"
].iloc[0]

low_visibility_rate = visibility_table.loc[
    visibility_table["visibility_bucket"] == "below_500_impressions",
    "declining_rate"
].iloc[0]

visibility_difference = visible_rate - low_visibility_rate

if visibility_difference > 0.05:
    visibility_verdict = "CONFIRMED"
elif visibility_difference < -0.05:
    visibility_verdict = "OPPOSITE"
else:
    visibility_verdict = "MIXED"

print(f"Visibility verdict: {visibility_verdict}")
print(f"Observed declining-rate difference: {visibility_difference:.3f}")

print("\nSignal audit complete.")
print("Important: trend_direction was used only to audit the signals;")
print("it is not used by the baseline score.")

Dataset: /content/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44

=== SIGNAL 1: STALENESS ===
       staleness_bucket     n  declining_n  declining_rate
    stale_180_plus_days   174           82        0.471264
updated_within_180_days 29826        16180        0.542480
Staleness verdict: OPPOSITE
Observed declining-rate difference: -0.071

=== SIGNAL 2: VISIBILITY ===
           visibility_bucket     n  declining_n  declining_rate
       below_500_impressions 13274         6301        0.474687
visible_500_plus_impressions 16726         9961        0.595540
Visibility verdict: CONFIRMED
Observed declining-rate difference: 0.121

Signal audit complete.
Important: trend_direction was used only to audit the signals;
it is not used by the baseline score.


## 2. Build the ranked queue (writes the CSV)

The baseline score is:

`stale × visible × impressions_90d`

where:

- `stale = 1` when `days_since_last_update >= 180`;
- `visible = 1` when `impressions_90d >= 500`.

Therefore:

- a stale and visible page receives its impressions as the score;
- every other page receives score `0`.

The score is transparent and uses only information available in the dataset before the review decision.

The queue is ranked from highest score to lowest score. Selected pages receive the reason code `stale_visible_page` and action `review_refresh`.

In [3]:
# Encode exactly ONE baseline rule.

stale = (
    df["days_since_last_update"] >= STALE_DAYS
).astype(int)

visible = (
    df["impressions_90d"] >= VISIBLE_IMPRESSIONS
).astype(int)

df["baseline_score"] = (
    stale
    * visible
    * df["impressions_90d"]
)

df["reason_code"] = np.where(
    df["baseline_score"] > 0,
    "stale_visible_page",
    "not_selected"
)

df["action"] = np.where(
    df["baseline_score"] > 0,
    "review_refresh",
    "monitor"
)

queue = (
    df.sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Total pages:", len(queue))
print("Pages selected for review:", int((queue["baseline_score"] > 0).sum()))

display(
    queue[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "days_since_last_update",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

# Write the required artifact.
output_dir = os.path.join(
    os.path.dirname(DATA_PATH),
    "..",
    "..",
    "work",
    "outputs"
)
output_dir = os.path.abspath(output_dir)
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

output_columns = [
    "rank",
    "content_id",
    "impressions_90d",
    "days_since_last_update",
    "baseline_score",
    "reason_code",
    "action"
]

queue[output_columns].to_csv(
    output_path,
    index=False
)

print("\nWrote:", output_path)
print("CSV rows:", len(queue))

Total pages: 30000
Pages selected for review: 17


,rank,content_id,impressions_90d,days_since_last_update,baseline_score,reason_code,action
0,1,content_cf56e2e2e282,61678,194,61678,stale_visible_page,review_refresh
1,2,content_7368877ea310,59472,194,59472,stale_visible_page,review_refresh
2,3,content_1bfaa38ff26c,25715,194,25715,stale_visible_page,review_refresh
3,4,content_0a91db491d14,13299,193,13299,stale_visible_page,review_refresh
4,5,content_5feee3994adb,7812,194,7812,stale_visible_page,review_refresh
5,6,content_c2d929d83eaa,7558,193,7558,stale_visible_page,review_refresh
6,7,content_b16bd7307b39,4590,194,4590,stale_visible_page,review_refresh
7,8,content_fe16a55cd13d,4556,194,4556,stale_visible_page,review_refresh
8,9,content_ecb6215e79fd,4429,194,4429,stale_visible_page,review_refresh
9,10,content_928af3e22c80,1697,193,1697,stale_visible_page,review_refresh



Wrote: /content/work/outputs/baseline_action_score.csv
CSV rows: 30000


## 3. Top-20 review

The top 20 are a **human review queue**, not automatic refresh instructions.

For each recommendation, I record:

- **action** — what the rule asks a reviewer to do;
- **reason code** — why the page was selected;
- **confidence note** — how strongly the rule supports the recommendation;
- **what would make it wrong** — evidence that could invalidate the recommendation.

The rule is intentionally narrow. A page can be old and visible without actually needing a content refresh.

In [4]:
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["baseline_score"] >= queue["baseline_score"].quantile(0.95):
        return (
            "High rule confidence: it is stale, highly visible, "
            "and sits near the top of this rule's ranking."
        )
    elif row["baseline_score"] >= queue["baseline_score"].quantile(0.75):
        return (
            "Moderate rule confidence: it satisfies both conditions "
            "and has meaningful visibility, but the rule is still narrow."
        )
    else:
        return (
            "Lower rule confidence: it satisfies the rule, but its "
            "visibility is weaker than the highest-ranked candidates."
        )

def wrong_reason(row):
    return (
        "It could be wrong if the page is intentionally evergreen, "
        "the update-date field is incomplete, demand is seasonal, "
        "or a refresh would not address the page's actual issue."
    )

top20_review = top20[
    [
        "rank",
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

top20_review["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

display(top20_review)

,rank,content_id,impressions_90d,days_since_last_update,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,61678,194,61678,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
1,2,content_7368877ea310,59472,194,59472,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
2,3,content_1bfaa38ff26c,25715,194,25715,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
3,4,content_0a91db491d14,13299,193,13299,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
4,5,content_5feee3994adb,7812,194,7812,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
5,6,content_c2d929d83eaa,7558,193,7558,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
6,7,content_b16bd7307b39,4590,194,4590,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
7,8,content_fe16a55cd13d,4556,194,4556,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
8,9,content_ecb6215e79fd,4429,194,4429,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...
9,10,content_928af3e22c80,1697,193,1697,stale_visible_page,review_refresh,"High rule confidence: it is stale, highly visi...",It could be wrong if the page is intentionally...


## 4. Weak picks + leakage check

The weak-pick review challenges the bottom of the selected queue rather than assuming every rule match is useful.

A weak pick can be stale and visible but still be a poor refresh candidate because of seasonality, intentional evergreen content, incomplete freshness metadata, or another page absorbing demand.

The leakage check confirms that the baseline does not use product decision flags, `trend_direction`, `trend_pct`, or future-window information.

In [5]:
selected = queue[queue["baseline_score"] > 0].copy()

print("=== WEAK PICKS ===")

if len(selected) == 0:
    print("No pages satisfy the baseline rule, so there are no selected weak picks to inspect.")
else:
    weak_picks = selected.tail(min(10, len(selected)))

    weak_review = weak_picks[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "days_since_last_update",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].copy()

    weak_review["why_it_may_be_weak"] = (
        "It matches the stale+visible rule, but the rule does not "
        "know whether a refresh would actually improve performance."
    )

    display(weak_review)

# ------------------------------------------------------------
# Leakage / forbidden-input check
# ------------------------------------------------------------

baseline_inputs = {
    "days_since_last_update",
    "impressions_90d"
}

forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "refresh_flag",
    "quick_win_flag"
}

overlap = baseline_inputs.intersection(forbidden_inputs)

print("\n=== LEAKAGE CHECK ===")
print("Baseline inputs:", sorted(baseline_inputs))
print("Forbidden / label-derived inputs:", sorted(forbidden_inputs))
print("Overlap:", sorted(overlap))

assert not overlap, (
    f"Leakage detected in baseline inputs: {sorted(overlap)}"
)

print("Product decision flags used: NO")
print("Future-window inputs used: NO")
print("Label-derived inputs used by score: NO")
print("Leakage check: PASS")

# Explicitly verify that trend_direction does not change the score.
score_without_label = (
    (df["days_since_last_update"] >= STALE_DAYS).astype(int)
    *
    (df["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)
    *
    df["impressions_90d"]
)

assert np.array_equal(
    df["baseline_score"].to_numpy(),
    score_without_label.to_numpy()
)

print("Score independence from trend_direction: PASS")

=== WEAK PICKS ===


,rank,content_id,impressions_90d,days_since_last_update,baseline_score,reason_code,action,why_it_may_be_weak
7,8,content_fe16a55cd13d,4556,194,4556,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
8,9,content_ecb6215e79fd,4429,194,4429,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
9,10,content_928af3e22c80,1697,193,1697,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
10,11,content_e3ff1b093148,1408,183,1408,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
11,12,content_bdbec75c1148,1316,194,1316,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
12,13,content_7f116ae1f6f5,954,301,954,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
13,14,content_77d4d5930e5e,828,194,828,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
14,15,content_72496874f806,821,301,821,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
15,16,content_6226ee6adc91,545,183,545,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."
16,17,content_074ba6ead17b,533,183,533,stale_visible_page,review_refresh,"It matches the stale+visible rule, but the rul..."



=== LEAKAGE CHECK ===
Baseline inputs: ['days_since_last_update', 'impressions_90d']
Forbidden / label-derived inputs: ['action_type', 'health_score', 'needs_ctr_fix', 'priority_score', 'quick_win_flag', 'refresh_flag', 'trend_direction', 'trend_pct']
Overlap: []
Product decision flags used: NO
Future-window inputs used: NO
Label-derived inputs used by score: NO
Leakage check: PASS
Score independence from trend_direction: PASS


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.